In [ ]:
# --- change file name ---

from pathlib import Path

# tg to tsv
sub_list = ['sub-01','sub-02','sub-03']
tg_dir = Path(r'X:\data\Articulation\SequenceFingerSpeech\SFS2\data_TextGrid\manual')

for sub in sub_list:
    folder = tg_dir / sub

    # Find all .TextGrid files
    for file in folder.glob("*.TextGrid"):
        if "_man" in file.stem:
            new_name = file.name.replace("_man", "")
            new_path = file.with_name(new_name)

            file.rename(new_path)
            print(f"Renamed: {file.name} -> {new_name}")

Renamed: sub-01_run-02_trial-01_man.TextGrid -> sub-01_run-02_trial-01.TextGrid
Renamed: sub-01_run-02_trial-02_man.TextGrid -> sub-01_run-02_trial-02.TextGrid
Renamed: sub-01_run-02_trial-05_man.TextGrid -> sub-01_run-02_trial-05.TextGrid
Renamed: sub-01_run-02_trial-08_man.TextGrid -> sub-01_run-02_trial-08.TextGrid
Renamed: sub-01_run-02_trial-09_man.TextGrid -> sub-01_run-02_trial-09.TextGrid
Renamed: sub-01_run-02_trial-12_man.TextGrid -> sub-01_run-02_trial-12.TextGrid
Renamed: sub-01_run-02_trial-13_man.TextGrid -> sub-01_run-02_trial-13.TextGrid
Renamed: sub-01_run-02_trial-14_man.TextGrid -> sub-01_run-02_trial-14.TextGrid
Renamed: sub-01_run-02_trial-17_man.TextGrid -> sub-01_run-02_trial-17.TextGrid
Renamed: sub-01_run-02_trial-18_man.TextGrid -> sub-01_run-02_trial-18.TextGrid
Renamed: sub-01_run-02_trial-19_man.TextGrid -> sub-01_run-02_trial-19.TextGrid
Renamed: sub-01_run-02_trial-23_man.TextGrid -> sub-01_run-02_trial-23.TextGrid
Renamed: sub-01_run-02_trial-25_man.Text

In [ ]:
# --- convert text grid to tsv ---

import os
from SylSeg.utils import TextGrids_to_tsv

# tg to tsv
sub_list = ['sub-01','sub-02','sub-03']
tg_dir = r'X:\data\Articulation\SequenceFingerSpeech\SFS2\data_TextGrid\manual'

for sub in sub_list:
    TEXTGRID_FOLDER = os.path.join(tg_dir,sub)
    # Output TSV
    OUTPUT_TSV_PATH = os.path.join(tg_dir,sub,f'{sub}.tsv')
    
    # Convert
    TextGrids_to_tsv.textgrids_to_output_tsv(
        textgrid_folder=TEXTGRID_FOLDER,
        output_tsv_path=OUTPUT_TSV_PATH,
        silence_label="silent",)


Found 206 TextGrid files.
[1/206] Processing: sub-01_run-02_trial-01.TextGrid
    Found 13 non-silent intervals.
[2/206] Processing: sub-01_run-02_trial-02.TextGrid
    Found 13 non-silent intervals.
[3/206] Processing: sub-01_run-02_trial-05.TextGrid
    Found 14 non-silent intervals.
[4/206] Processing: sub-01_run-02_trial-08.TextGrid
    Found 13 non-silent intervals.
[5/206] Processing: sub-01_run-02_trial-09.TextGrid
    Found 13 non-silent intervals.
[6/206] Processing: sub-01_run-02_trial-12.TextGrid
    Found 13 non-silent intervals.
[7/206] Processing: sub-01_run-02_trial-13.TextGrid
    Found 5 non-silent intervals.
[8/206] Processing: sub-01_run-02_trial-14.TextGrid
    Found 14 non-silent intervals.
[9/206] Processing: sub-01_run-02_trial-17.TextGrid
    Found 14 non-silent intervals.
[10/206] Processing: sub-01_run-02_trial-18.TextGrid
    Found 13 non-silent intervals.
[11/206] Processing: sub-01_run-02_trial-19.TextGrid
    Found 13 non-silent intervals.
[12/206] Proces

In [ ]:
# --- add is_syllable column ---

import pandas as pd
import os


# tg to tsv
sub_list = ['sub-01','sub-02','sub-03']
tg_dir = r'X:\data\Articulation\SequenceFingerSpeech\SFS2\data_TextGrid\manual'
is_syllable = ['1', '2', '3', '4', '5']

for sub in sub_list:
    
    # load tsv file
    tsv_file = os.path.join(tg_dir,sub,f"{sub}.tsv")
    df = pd.read_csv(tsv_file, sep='\t')

    # Add column
    df['is_syllable'] = df['syllable'].isin(is_syllable).astype(int)
    
    # Recalculate syllable_number (bell = 0, first syllable = 1, etc.)
    group_cols = ['subject', 'run', 'trial']

    df['syllable_number'] = (df.groupby(group_cols)['is_syllable'].cumsum())
    
    # Set non-syllable events to 0
    df.loc[df['is_syllable'] == 0, 'syllable_number'] = 0

    # Save updated TSV
    df.to_csv(tsv_file, sep='\t', index=False)

In [ ]:

# -- add data to the orig .dat file

import pandas as pd
from pathlib import Path


# Directories
base_dir = Path(r'X:\data\Articulation\SequenceFingerSpeech\SFS2')
tg_dir = base_dir / 'data_TextGrid' / 'manual'
raw_dir = base_dir / 'data_raw'

sub_list = ['sub-01', 'sub-02', 'sub-03']

group_cols = ['subject', 'run', 'trial']

for sub in sub_list:

    print(f'\nProcessing {sub}...')

    # ---------------------------------
    # Load TextGrid TSV
    # ---------------------------------

    tsv_file = tg_dir / sub / f'{sub}.tsv'
    df = pd.read_csv(tsv_file, sep='\t')

    # ---------------------------------
    # Load behavioral TSV
    # ---------------------------------

    behav_file = raw_dir / sub / f'{sub}.tsv'
    behav = pd.read_csv(behav_file, sep='\t')

    # ---------------------------------
    # Calculate RT, ET, MT per trial
    # ---------------------------------

    results = []

    for (subject, run, trial), group in df.groupby(group_cols):

        bell = group[group['is_syllable'] == 0]
        syllables = group[group['is_syllable'] == 1]

        if bell.empty or syllables.empty:
            print(f'Warning: Missing bell or syllables: {subject}, run {run}, trial {trial}')
            continue

        # First bell onset
        bell_onset = bell['onset_sec'].min()

        # First syllable onset
        first_syll_onset = syllables['onset_sec'].min()

        # Last syllable offset
        last_syll_offset = syllables['offset_sec'].max()

        results.append({
            'subject': subject,
            'run': run,
            'trial': trial,
            'RT': first_syll_onset - bell_onset,
            'ET': last_syll_offset - bell_onset,
            'MT': last_syll_offset - first_syll_onset
        })

    trial_times = pd.DataFrame(results)

    # ---------------------------------
    # Update existing RT, ET, MT columns
    # ---------------------------------

    for _, row in trial_times.iterrows():

        mask = (
            (behav['subject'] == row['subject']) &
            (behav['run'] == row['run']) &
            (behav['trial'] == row['trial'])
        )

        behav.loc[mask, ['RT', 'ET', 'MT']] = [
            row['RT'],
            row['ET'],
            row['MT']
        ]

    # ---------------------------------
    # Update pressTime1-pressTime11
    # Each pressTime = syllable duration
    # ---------------------------------

    for _, row in df[df['is_syllable'] == 1].iterrows():

        syll_num = int(row['syllable_number'])

        if 1 <= syll_num <= 11:

            mask = (
                (behav['subject'] == row['subject']) &
                (behav['run'] == row['run']) &
                (behav['trial'] == row['trial'])
            )

            duration = row['offset_sec'] - row['onset_sec']

            behav.loc[mask, f'pressTime{syll_num}'] = duration

    # ---------------------------------
    # Rename columns
    # ---------------------------------

    behav = behav.rename(columns={
        'run': 'BN',
        'trial': 'TN',
        'subject': 'SubNum'
    })

# ---------------------------------
    # Save as a new TSV
    # ---------------------------------

    output_file = raw_dir / sub / f'{sub}_full.tsv'

    behav.to_csv(output_file, sep='\t', index=False)

    print(f'Saved: {output_file}')



Processing sub-01...
Saved: X:\data\Articulation\SequenceFingerSpeech\SFS2\data_raw\sub-01\sub-01_full.tsv

Processing sub-02...
Saved: X:\data\Articulation\SequenceFingerSpeech\SFS2\data_raw\sub-02\sub-02_full.tsv

Processing sub-03...
Saved: X:\data\Articulation\SequenceFingerSpeech\SFS2\data_raw\sub-03\sub-03_full.tsv

Done!
